In [ ]:
# Series & DataFrame

In [8]:
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Dataset 1: Server Metrics ---
servers     = ["srv-01", "srv-02", "srv-03", "srv-04", "srv-05"]
timestamps  = pd.date_range("2026-01-01", periods=100, freq="1h")

records = []
for ts in timestamps:
    for srv in servers:
        records.append({
            "timestamp":   ts,
            "server_id":   srv,
            "cpu_pct":     round(np.random.uniform(20, 99), 1),
            "memory_pct":  round(np.random.uniform(30, 98), 1),
            "response_ms": round(np.random.uniform(50, 1200), 1),
            "disk_pct":    round(np.random.uniform(30, 95), 1),
            "status":      np.random.choice(["ok","warning","critical"], p=[0.75, 0.20, 0.05])
        })

df_metrics = pd.DataFrame(records)
df_metrics.to_csv("server_metrics.csv", index=False)
print("server_metrics.csv saved →", df_metrics.shape)


# --- Dataset 2: Incident Tickets ---
categories = ["CPU High", "Disk Full", "Network Lag", "Memory Leak", "Service Down"]
priorities = [1, 2, 3]

tickets = []
for i in range(200):
    created = pd.Timestamp("2026-01-01") + pd.Timedelta(hours=np.random.randint(0, 2400))
    resolved = created + pd.Timedelta(hours=np.random.randint(1, 72))
    is_resolved = np.random.choice([True, False], p=[0.65, 0.35])
    tickets.append({
        "ticket_id":   f"INC{str(i+1).zfill(4)}",
        "server_id":   np.random.choice(servers),
        "category":    np.random.choice(categories),
        "priority":    np.random.choice(priorities, p=[0.3, 0.5, 0.2]),
        "created_at":  created,
        "resolved_at": resolved if is_resolved else pd.NaT,
        "resolved":    is_resolved
    })

df_tickets = pd.DataFrame(tickets)
df_tickets.to_csv("incidents.csv", index=False)
print("incidents.csv saved →", df_tickets.shape)


# --- Dataset 3: Application Logs ---
log_levels = ["INFO", "WARNING", "ERROR", "CRITICAL"]
messages = [
    "CPU threshold exceeded",
    "Disk usage at 90%",
    "Connection timeout error",
    "Memory allocation failed",
    "Service restarted successfully",
    "Network packet loss detected",
    "Database query slow",
    "Authentication failed",
]

logs = []
for i in range(300):
    ts = pd.Timestamp("2026-01-01") + pd.Timedelta(minutes=np.random.randint(0, 14400))
    logs.append({
        "timestamp":   ts,
        "server_id":   np.random.choice(servers),
        "log_level":   np.random.choice(log_levels, p=[0.5, 0.25, 0.15, 0.10]),
        "message":     np.random.choice(messages),
        "error_code":  np.random.choice(["E001","E002","E003","E004", None], p=[0.2,0.2,0.2,0.2,0.2])
    })

df_logs = pd.DataFrame(logs)
df_logs.to_csv("app_logs.csv", index=False)
print("app_logs.csv saved →", df_logs.shape)

server_metrics.csv saved → (500, 7)
incidents.csv saved → (200, 7)
app_logs.csv saved → (300, 5)


In [9]:
# Series - one metric across fleet of servers
import pandas as pd
cpu = pd.Series(
    [45,78,30,92,55], 
    index = ["srv-01", "srv-02", "srv-03", "srv-04", "srv-05"],
    name="cpu_pct"
)
print(cpu)

srv-01    45
srv-02    78
srv-03    30
srv-04    92
srv-05    55
Name: cpu_pct, dtype: int64


In [12]:
# Series operations - instant servers health
print(cpu.mean())
print(cpu.max())
print(cpu.min())
print(cpu.std())
print(cpu.sum())
print(cpu[cpu>80])

60.0
92
30
24.9899979991996
300
srv-04    92
Name: cpu_pct, dtype: int64


In [33]:
# DataFrames - full server snapshot
data = {
    "server_id": ["srv-01", "srv-02", "srv-03", "srv-04", "srv-05"],
    "cpu_pct" : [45,78,30,92,55],
    "memory_pct" : [60,82,40,95,65],
    "response_ms" : [120,340,89, 950, 210],
    "disk_pct" : [55,70,45,88,60],
    "status" : ["ok", "warning", "ok", "critical", "ok"]
}
df = pd.DataFrame(data)
print(df)

  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status
0    srv-01       45          60          120        55        ok
1    srv-02       78          82          340        70   warning
2    srv-03       30          40           89        45        ok
3    srv-04       92          95          950        88  critical
4    srv-05       55          65          210        60        ok


In [34]:
# DataFrame from list of dicts - API Ingestion pattern
incidents = [
    {"ticket_id" : "INC001", "server": "srv-04", "category" : "CPU Hight", "priority" : 1, "resolved": False},
    {"ticket_id" : "INC002", "server" : "srv-02", "category" : "Disk Full", "priority" : 2, "resolved": False},
    {"ticket_id" : "INC003", "server" : "srv-01", "category" : "Network Lag", "priority" : 3, "resolved" : True},
    {"ticket_id" : "INC004", "server" : "srv-04", "category" : "Memory Leak", "priority" : 1, "resolved": False},
]
df_inc = pd.DataFrame(incidents)
print(df_inc)

  ticket_id  server     category  priority  resolved
0    INC001  srv-04    CPU Hight         1     False
1    INC002  srv-02    Disk Full         2     False
2    INC003  srv-01  Network Lag         3      True
3    INC004  srv-04  Memory Leak         1     False


In [38]:
# Accessing columns

df["cpu_pct"] # single column - series

df[["server_id", "disk_pct", "status"]] # multiple columns - DataFrame

,server_id,disk_pct,status
0,srv-01,55,ok
1,srv-02,70,warning
2,srv-03,45,ok
3,srv-04,88,critical
4,srv-05,60,ok
